## Install missing packages

In [ ]:
pip install lmfit

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import lmfit

## Read Nexus file

In [ ]:
file = h5py.File(os.getcwd() + "/IV_temp.nxs")

# Each row is one I-V sweep at a fixed temperature
voltages = np.array(file['/entry/data/voltage'])
currents = np.array(file['/entry/data/current'])
temperatures = np.array(file['/entry/data/temperature'])

n_sweeps, n_points = currents.shape
voltages = voltages.reshape(n_sweeps, n_points)
temperatures = temperatures.reshape(n_sweeps, n_points)

## Run a simple fitting and calculate the resistance

In [ ]:
stdColors = plt.rcParams['axes.prop_cycle'].by_key()['color']
fitmodel = lmfit.models.LinearModel()

resistances, sweep_temps = [], []
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for i in range(n_sweeps):
    v, c = voltages[i], currents[i]
    t_mean = temperatures[i].mean()
    sweep_temps.append(t_mean)

    ax1.plot(v, c, 'x', label=f'T = {t_mean:.1f} K', color=stdColors[i])
    result = fitmodel.fit(c, x=v)
    ax1.plot(v, result.best_fit, color=stdColors[i])

    resistances.append(result.params['slope'].value)

ax1.legend()
ax1.set_xlabel('voltage (V)')
ax1.set_ylabel('current (A)')
ax1.set_title('I–V curves by temperature')

ax2.plot(np.array(sweep_temps), resistances, 'x')
ax2.set_xlabel('temperature (K)')
ax2.set_ylabel(r'resistance ($\Omega$)')
ax2.set_title('Resistance vs. temperature')

plt.tight_layout()